<a href="https://colab.research.google.com/github/aniget/SoftUni-AI-Integrations-for-developers/blob/main/AI_Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 AI Learning Assistant — RAG-Powered Educational Tutor

> **Built with:** Claude API · ChromaDB · SentenceTransformers · PyMuPDF · gTTS

---

## What This Notebook Teaches

This notebook is intentionally **dual-purpose**:

| Audience | What you'll get |
|---|---|
| 🧒 **Students (end users)** | An AI tutor that reads your PDF textbook and answers questions in your language, at your age level |
| 🎓 **Engineering students** | A hands-on walkthrough of RAG architecture, vector databases, embedding models, and LLM prompt engineering |

---

## System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                        USER INPUT                           │
│              (text question + age + language)               │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                    ask_ai(question)                         │
│         Structured output parsing via Claude API            │
│         → extracts: { prompt, format }                      │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│              retrieve_information(prompt)                   │
│  1. Embed question   →  SentenceTransformer                 │
│  2. Similarity search →  ChromaDB (top-k chunks)            │
│  3. Build context    →  assembled prompt                     │
│  4. Generate answer  →  Claude API                          │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                   OUTPUT ROUTER                             │
│   text  →  display in notebook                              │
│   audio →  gTTS .mp3 with playback widget                   │
│   image →  Claude-generated DALL·E-style prompt → display   │
└─────────────────────────────────────────────────────────────┘
```

---

## Key Engineering Concepts Covered

- **Chunking strategy** — why size and overlap matter for retrieval quality
- **Dense embeddings** — how text becomes vectors, and why similarity search works
- **Vector databases** — ChromaDB internals and the HNSW index
- **Prompt engineering** — structured output, anti-hallucination, age adaptation
- **RAG vs. fine-tuning** — when to use which approach
- **Multimodal output routing** — intent detection and output dispatch


---
## 📦 Section 1 — Environment Setup

### Engineering Note: Dependency Management

We install exactly what we need. Each library serves a specific role:

| Library | Role in the system |
|---|---|
| `anthropic` | Claude API client — our LLM backbone |
| `chromadb` | In-memory vector store with HNSW indexing |
| `sentence-transformers` | Local embedding model (no API cost per embedding) |
| `pymupdf` (fitz) | Fast, accurate PDF text extraction |
| `gtts` | Google Text-to-Speech — converts answer to audio |
| `Pillow` | Image display utilities |

> **Why use a local embedding model instead of an API?**  
> Embedding is called for every chunk during indexing AND every query at runtime.  
> A local model like `all-MiniLM-L6-v2` runs in ~5ms per chunk with zero cost, while API embeddings add latency and cost at scale.


In [ ]:
# ─── Install all required dependencies ───────────────────────────────────────
# Run this cell once. Colab's environment resets per session,
# so this must be re-run if the runtime is restarted.

!pip install -q anthropic chromadb sentence-transformers pymupdf gtts Pillow requests

print("✅ All dependencies installed successfully.")

In [ ]:
# ─── Standard Library Imports ─────────────────────────────────────────────────
import os
import json
import re
import textwrap
import tempfile
from pathlib import Path
from typing import Literal, Optional
from dataclasses import dataclass

# ─── Third-Party Imports ──────────────────────────────────────────────────────
import fitz                                          # PyMuPDF — PDF parsing
import chromadb                                      # Vector store
from sentence_transformers import SentenceTransformer  # Local embedding model
from gtts import gTTS                                # Text-to-Speech
import anthropic                                     # Claude API client

# ─── Colab / IPython Display Utilities ───────────────────────────────────────
from IPython.display import display, Audio, Markdown, Image, HTML
from google.colab import files, userdata

print("✅ All imports successful.")

;lk;k;k;k;lk# 🔑 Section 2 — API Key Configuration

### Engineering Note: Secret Management

**Never hardcode API keys in source code.** This is a critical security principle.

In Colab, the recommended approach is **Colab Secrets** (the 🔑 icon in the left sidebar).  
This stores keys encrypted and accessible only to your account.

We fall back to `os.getenv()` for CI/CD or local environments.

```
Colab Secrets  →  userdata.get('ANTHROPIC_API_KEY')
Environment    →  os.getenv('ANTHROPIC_API_KEY')
```


In [ ]:
# ─── Secure API Key Loading ───────────────────────────────────────────────────
# Priority 1: Colab Secrets (recommended)
# Priority 2: Environment variable (for local / CI use)

def load_api_key() -> str:
    """Load the Anthropic API key from Colab Secrets or environment.

    Returns:
        The API key string.

    Raises:
        ValueError: If no API key is found in either location.
    """
    # Try Colab Secrets first
    try:
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            print("✅ API key loaded from Colab Secrets.")
            return key
    except Exception:
        pass

    # Fall back to environment variable
    key = os.getenv('ANTHROPIC_API_KEY')
    if key:
        print("✅ API key loaded from environment variable.")
        return key

    raise ValueError(
        "❌ No API key found.\n"
        "→ Add ANTHROPIC_API_KEY to Colab Secrets (🔑 in the sidebar)\n"
        "→ Or set it as an environment variable."
    )


ANTHROPIC_API_KEY = load_api_key()

# Initialise the Claude client — this object is reused throughout the notebook
claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

### HuggingFace Token — for Image Generation

A **free** HuggingFace account gives you access to the Inference API for image generation.

1. Register at [huggingface.co](https://huggingface.co) (free)
2. Go to **Settings → Access Tokens → New token** (read permission is enough)
3. Add it as a Colab Secret named `HF_TOKEN` (same 🔑 sidebar as before)

> **Free tier limits:** ~30,000 inference calls/month. More than enough for a notebook.


In [ ]:
# ─── HuggingFace Token Loader ─────────────────────────────────────────────────
# Required only for the image output feature.
# If you don't have a HF token, the system still works for text and audio.

def load_hf_token() -> str:
    """Load the HuggingFace API token from Colab Secrets or environment.

    Returns an empty string (not an error) if not found —
    image generation will fail gracefully with a helpful message.
    """
    try:
        key = userdata.get('HF_TOKEN')
        if key:
            print("✅ HuggingFace token loaded from Colab Secrets.")
            return key
    except Exception:
        pass

    key = os.getenv('HF_TOKEN', '')
    if key:
        print("✅ HuggingFace token loaded from environment variable.")
        return key

    print("⚠️  No HuggingFace token found. Image generation will not work.")
    print("   Text and audio output are unaffected.")
    return ""


---
## ⚙️ Section 3 — Global Configuration

### Engineering Note: Centralised Configuration

Separating configuration from logic is a key software engineering principle.  
A single `Config` dataclass means changing a parameter (e.g., chunk size) only requires editing **one line**, not hunting through functions.

**Key parameters to understand:**

- **`CHUNK_SIZE`**: How many tokens per text chunk. Too small → poor context for the LLM. Too large → retrieval becomes imprecise.
- **`CHUNK_OVERLAP`**: Overlap between adjacent chunks prevents important context from being split across boundaries.
- **`TOP_K`**: How many chunks to retrieve per query. More chunks = more context, but also more noise and token cost.
- **`EMBEDDING_MODEL`**: `all-MiniLM-L6-v2` is 80MB, fast, and surprisingly accurate for its size.


In [ ]:
# ─── Global Configuration ─────────────────────────────────────────────────────

@dataclass
class Config:
    """Centralised configuration for the AI Learning Assistant.

    Modify these values to experiment with the system's behaviour.
    All downstream functions read from this single source of truth.
    """
    # ── RAG Parameters ──────────────────────────────────────────────────────
    CHUNK_SIZE: int = 600          # Target token count per chunk (~4 chars/token)
    CHUNK_OVERLAP: int = 100       # Overlap tokens between adjacent chunks
    TOP_K: int = 4                 # Number of chunks to retrieve per query
    SIMILARITY_THRESHOLD: float = 0.3  # Min cosine similarity to include a chunk

    # ── Model Configuration ──────────────────────────────────────────────────
    EMBEDDING_MODEL: str = "all-MiniLM-L6-v2"   # Local embedding model
    CLAUDE_MODEL: str = "claude-haiku-4-5-20251001"      # Claude model for generation
    MAX_TOKENS: int = 1024                        # Max tokens in LLM response

    # ── ChromaDB Configuration ───────────────────────────────────────────────
    COLLECTION_NAME: str = "learning_assistant_chunks"

    # ── Output Configuration ─────────────────────────────────────────────────
    AUDIO_LANG_MAP: dict = None   # Maps language codes to gTTS codes

    # ── HuggingFace Image Generation ─────────────────────────────────────────
    # Token is loaded separately in load_hf_token() below.
    # To switch models, change HF_MODEL to any SD model on huggingface.co.
    # Good free alternatives:
    #   - "runwayml/stable-diffusion-v1-5"       (faster, smaller)
    #   - "black-forest-labs/FLUX.1-schnell"  (better quality)
    #   - "dreamlike-art/dreamlike-photoreal-2.0" (photorealistic)

    HF_MODEL: str = "black-forest-labs/FLUX.1-schnell"
    HF_TOKEN: str = ""   # Populated at runtime by load_hf_token()

    def __post_init__(self):
        self.AUDIO_LANG_MAP = {
            "en": "en",
            "bg": "bg",
        }


# ─── Age-to-Style Mapping ─────────────────────────────────────────────────────
# These descriptions are injected directly into the LLM prompt.
# Engineering note: this is a form of "soft" prompt routing —
# the same model produces different outputs based on instruction framing.

AGE_STYLE_MAP = {
    "7-10":  "Use very simple words, short sentences, fun analogies, and a storytelling style. "
             "Imagine you are explaining to a curious 8-year-old. Use emojis occasionally.",

    "10-16": "Use clear language with relatable real-world examples and analogies. "
             "Avoid jargon unless you explain it. Keep a friendly, encouraging tone.",

    "16+":   "Use structured explanations with proper terminology. "
             "You may include brief technical depth where appropriate.",

    "30+":   "Be concise and professional. Assume background knowledge. "
             "Offer optional deeper explanations where relevant.",
}


# Instantiate the global config object
cfg = Config()
cfg.HF_TOKEN =load_hf_token()

print("✅ Configuration ready.")
print(f"   Chunk size: {cfg.CHUNK_SIZE} tokens | Overlap: {cfg.CHUNK_OVERLAP} | Top-K: {cfg.TOP_K}")
print(f"   Embedding model: {cfg.EMBEDDING_MODEL}")
print(f"   Claude model: {cfg.CLAUDE_MODEL}")

---
## 📄 Section 4 — PDF Processing Pipeline

### Engineering Note: Why Chunking Strategy Matters

A PDF textbook might contain 50,000 words. An LLM context window holds ~4,000–200,000 tokens.  
Even with large context windows, **retrieving only relevant sections** is better because:

1. **Cost** — fewer tokens = lower API cost
2. **Accuracy** — a focused context reduces the chance the model is distracted by irrelevant text
3. **Speed** — smaller prompts are processed faster

**The overlap parameter** ensures that a sentence split across two chunks is fully captured by at least one.  
Without overlap, a question about content at a chunk boundary would retrieve incomplete context.

```
Chunk 1: [  ←── 600 tokens ──→  ]
Chunk 2:               [  ←── 600 tokens ──→  ]
         |←── 100 token overlap ──→|
```


In [ ]:
# ─── PDF Text Extraction ──────────────────────────────────────────────────────

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """Extract text from a PDF file, preserving page metadata.

    Uses PyMuPDF (fitz) which handles both text-based and some scanned PDFs.
    Each page is returned as a separate dict to preserve source attribution.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        List of dicts: [{"page": int, "text": str}, ...]
    """
    pages = []
    doc = fitz.open(pdf_path)

    print(f"📄 Opened PDF: {Path(pdf_path).name} ({len(doc)} pages)")

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")   # "text" mode = plain text, preserving layout
        text = re.sub(r'\s+', ' ', text).strip()   # Normalise whitespace

        if len(text) > 50:   # Skip near-empty pages (headers, images, etc.)
            pages.append({"page": page_num, "text": text})

    doc.close()
    print(f"✅ Extracted text from {len(pages)} non-empty pages.")
    return pages


# ─── Text Chunking ────────────────────────────────────────────────────────────

def chunk_text(pages: list[dict], chunk_size: int, overlap: int) -> list[dict]:
    """Split page texts into overlapping chunks for retrieval.

    Engineering note:
        We approximate token count using character count / 4
        (a reasonable heuristic for English and Slavic languages).
        Production systems use a proper tokenizer (e.g., tiktoken) for precision.

    Args:
        pages:      Output from extract_text_from_pdf.
        chunk_size: Target size of each chunk in approximate tokens.
        overlap:    Number of tokens to overlap between adjacent chunks.

    Returns:
        List of chunk dicts with text, page, and chunk_id.
    """
    chunks = []
    chunk_id = 0

    # Convert token counts to approximate character counts
    char_size = chunk_size * 4
    char_overlap = overlap * 4
    step = char_size - char_overlap   # How far to advance the window each step

    for page_data in pages:
        text = page_data["text"]
        page_num = page_data["page"]

        # Slide a window across the page text
        start = 0
        while start < len(text):
            end = min(start + char_size, len(text))
            chunk_text_content = text[start:end].strip()

            if len(chunk_text_content) > 100:   # Ignore trivially small chunks
                chunks.append({
                    "chunk_id": f"chunk_{chunk_id}",
                    "page": page_num,
                    "text": chunk_text_content,
                })
                chunk_id += 1

            if end == len(text):
                break
            start += step

    print(f"✅ Created {len(chunks)} chunks (avg ~{chunk_size} tokens each).")
    return chunks

---
## 🧠 Section 5 — Embeddings & Vector Store

### Engineering Note: How Embeddings Work

An **embedding** is a high-dimensional vector (384 numbers for `all-MiniLM-L6-v2`) that encodes the **semantic meaning** of text.  
Similar sentences produce vectors that are **close together** in this 384-dimensional space.

**Cosine similarity** measures the angle between two vectors:  
- `1.0` = identical meaning  
- `0.0` = completely unrelated  
- `-1.0` = opposite meaning (rare in practice)

### Engineering Note: ChromaDB Architecture

ChromaDB is a purpose-built vector database.  
Under the hood, it uses an **HNSW (Hierarchical Navigable Small World)** index — a graph-based structure  
that makes approximate nearest-neighbour search extremely fast, even with millions of vectors.

For this notebook, we use the **in-memory** client, which is reset when the runtime restarts.  
In production, you'd persist to disk: `chromadb.PersistentClient(path="./chroma_db")`.


In [ ]:
# ─── Embedding Model Loader ───────────────────────────────────────────────────

def load_embedding_model(model_name: str) -> SentenceTransformer:
    """Load and return the SentenceTransformer embedding model.

    Downloads the model on first run (~80MB) and caches it locally.
    Subsequent calls load from cache in seconds.

    Args:
        model_name: HuggingFace model identifier.

    Returns:
        Loaded SentenceTransformer model.
    """
    print(f"⬇️  Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)
    print(f"✅ Embedding model ready. Output dimension: {model.get_sentence_embedding_dimension()}")
    return model


# ─── ChromaDB Vector Store Builder ───────────────────────────────────────────

def build_vector_store(
    chunks: list[dict],
    embedding_model: SentenceTransformer,
    collection_name: str
) -> chromadb.Collection:
    """Embed all chunks and store them in a ChromaDB collection.

    This is the core indexing step. After this function returns,
    the system can answer questions without re-reading the PDF.

    Args:
        chunks:           List of chunk dicts from chunk_text().
        embedding_model:  Loaded SentenceTransformer model.
        collection_name:  Name for the ChromaDB collection.

    Returns:
        ChromaDB Collection object, ready for similarity search.
    """
    # Use in-memory ChromaDB for Colab (no disk persistence needed)
    chroma_client = chromadb.Client()

    # Delete existing collection if it exists (safe re-run behaviour)
    try:
        chroma_client.delete_collection(name=collection_name)
    except Exception:
        pass

    collection = chroma_client.create_collection(
        name=collection_name,
        # ChromaDB default metric is cosine similarity — ideal for embeddings
        metadata={"hnsw:space": "cosine"}
    )

    print(f"🔄 Embedding {len(chunks)} chunks... (this may take 30–60 seconds)")

    # Extract text content for batch embedding
    texts = [c["text"] for c in chunks]

    # Batch embed — SentenceTransformer handles batching internally.
    # show_progress_bar=True gives a nice tqdm bar for large PDFs.
    # Note: we call .tolist() to convert the numpy array to plain Python lists,
    # which is what ChromaDB expects. Older ST versions had convert_to_list=True
    # as a parameter, but it was removed — .tolist() is the correct approach.
    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True,
    ).tolist()

    # Add all chunks to ChromaDB in a single batch operation
    collection.add(
        ids=[c["chunk_id"] for c in chunks],
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"page": c["page"]} for c in chunks]
    )

    print(f"✅ Vector store built with {collection.count()} vectors.")
    return collection

---
## 🔍 Section 6 — Retrieval & Answer Generation

### Engineering Note: The Retrieval-Augmented Generation (RAG) Pattern

RAG is the most important pattern in applied LLM engineering. Instead of relying on the model's  
parametric memory (which may be outdated or hallucinated), we:

1. **Retrieve** relevant text from a trusted source (the uploaded PDF)
2. **Augment** the prompt with that retrieved text
3. **Generate** a grounded answer that cites its sources

### Engineering Note: Anti-Hallucination via Prompt Design

The most reliable way to reduce hallucination is **explicit instruction in the system prompt**:  
- Tell the model exactly what to do when context is missing  
- Require source attribution with visual markers (`📄` vs `🌐`)  
- Use the phrase *"only based on the provided context"* — this anchors the model

This is not foolproof, but significantly reduces confabulation in practice.


In [ ]:
# ─── Context Retrieval ────────────────────────────────────────────────────────

def retrieve_chunks(
    query: str,
    collection: chromadb.Collection,
    embedding_model: SentenceTransformer,
    top_k: int
) -> list[dict]:
    """Embed a query and retrieve the most semantically similar chunks.

    Engineering note:
        ChromaDB returns results sorted by similarity (most similar first).
        The 'distances' field contains cosine distances (lower = more similar).
        We convert: similarity = 1 - distance

    Args:
        query:           The user's question.
        collection:      ChromaDB collection to search.
        embedding_model: Model to embed the query.
        top_k:           Number of chunks to retrieve.

    Returns:
        List of retrieved chunk dicts with text, page, and similarity score.
    """
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": doc,
            "page": meta["page"],
            "similarity": round(1 - dist, 4)   # Convert distance → similarity
        })

    return retrieved


# ─── Context Assembly ─────────────────────────────────────────────────────────

def assemble_context(chunks: list[dict]) -> str:
    """Format retrieved chunks into a readable context block for the LLM prompt.

    Engineering note:
        The structure and labelling of context significantly affects LLM behaviour.
        Clearly marking where each chunk comes from helps the model:
        1. Attribute answers correctly
        2. Notice when chunks are from different parts of the document
        3. Identify gaps in the retrieved context

    Args:
        chunks: Retrieved chunks from retrieve_chunks().

    Returns:
        Formatted context string for injection into the LLM prompt.
    """
    if not chunks:
        return "[No relevant context found in the document.]"

    parts = []
    for i, chunk in enumerate(chunks, 1):
        parts.append(
            f"--- Context Block {i} (Page {chunk['page']}, "
            f"relevance: {chunk['similarity']:.2%}) ---\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(parts)


# ─── Answer Generation via Claude API ────────────────────────────────────────

def generate_answer(
    question: str,
    context: str,
    age_group: str,
    language: str,
    client: anthropic.Anthropic
) -> str:
    """Generate a grounded, age-adapted answer using the Claude API.

    This function implements the core RAG pattern:
    - The SYSTEM prompt defines the model's role and constraints (anti-hallucination)
    - The USER prompt contains the retrieved context + the actual question

    Engineering note on prompt structure:
        Separating system instructions from user content is crucial.
        The system prompt sets persistent behavioural rules;
        the user message provides the dynamic, per-query content.

    Args:
        question:   The user's question.
        context:    Assembled retrieved context from assemble_context().
        age_group:  Key into AGE_STYLE_MAP (e.g., "7-10").
        language:   Language code ("en" or "bg").
        client:     Anthropic client instance.

    Returns:
        The LLM's text response.
    """
    style_instruction = AGE_STYLE_MAP.get(age_group, AGE_STYLE_MAP["16+"])
    lang_instruction = "Bulgarian" if language == "bg" else "English"

    system_prompt = f"""\
You are an expert, patient, and engaging educational tutor.

## Your Role
You answer questions ONLY based on the provided document context.
You adapt your language style to the student's age group.
You always respond in {lang_instruction}.

## Communication Style
{style_instruction}

## Source Attribution Rules (CRITICAL)
You MUST clearly label the origin of every piece of information:
- Use "📄 From the document:" for information found in the provided context
- Use "🌐 Additional context:" for any supplementary explanation you add
- If the answer is NOT in the provided context, say:
  "I couldn't find this in the uploaded material. 🌐 Here is what I know generally: ..."

## Anti-Hallucination Rules
- Do NOT fabricate page numbers or quotes not present in the context
- Do NOT claim certainty about information not in the context
- If context is partial, acknowledge what is and isn't covered
"""

    user_message = f"""\
## Retrieved Document Context
{context}

## Student Question
{question}

Please answer the question based on the context above.
Clearly mark whether each part of your answer comes from 📄 the document or 🌐 your general knowledge.
"""

    response = client.messages.create(
        model=cfg.CLAUDE_MODEL,
        max_tokens=cfg.MAX_TOKENS,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_message}
        ]
    )

    return response.content[0].text


# ─── Core RAG Function ────────────────────────────────────────────────────────

def retrieve_information(
    question: str,
    collection: chromadb.Collection,
    embedding_model: SentenceTransformer,
    age_group: str,
    language: str,
    client: anthropic.Anthropic,
    verbose: bool = False
) -> str:
    """End-to-end RAG pipeline: retrieve relevant chunks → generate grounded answer.

    This is the central function that wires together all components.
    Think of it as the 'retrieval + generation' half of the system.

    Args:
        question:        The user's question.
        collection:      The ChromaDB vector store.
        embedding_model: Loaded SentenceTransformer.
        age_group:       Age group key for style adaptation.
        language:        "en" or "bg".
        client:          Anthropic API client.
        verbose:         If True, prints retrieved chunks for debugging.

    Returns:
        The generated answer as a string.
    """
    # Step 1: Retrieve relevant chunks
    chunks = retrieve_chunks(question, collection, embedding_model, cfg.TOP_K)

    if verbose:
        print("\n🔍 Retrieved Chunks:")
        for i, c in enumerate(chunks, 1):
            print(f"  [{i}] Page {c['page']} | Similarity: {c['similarity']:.2%}")
            print(f"      {c['text'][:120]}...")
        print()

    # Step 2: Assemble context block
    context = assemble_context(chunks)

    # Step 3: Generate answer with Claude
    answer = generate_answer(question, context, age_group, language, client)

    return answer

---
## 🎛️ Section 7 — Output Routing & Multimodal Handlers

### Engineering Note: Intent Detection via Structured Output

Before routing to text/audio/image, we need to detect **what format the user wants**.  
We do this by asking Claude to parse the question into a structured JSON object:

```json
{"prompt": "<cleaned question>", "format": "text | audio | image"}
```

This is called **structured output** — a pattern where you constrain the LLM's output  
to a specific schema. This is far more robust than regex-matching the raw question.

**Why not just regex?**  
Natural language is ambiguous. "Tell me about X" in Bulgarian starts with "Кажи ми" or "Разкажи ми".  
A language model handles all variations and edge cases gracefully.


In [ ]:
# ─── Structured Output: Intent Detection ─────────────────────────────────────

def parse_intent(question: str, client: anthropic.Anthropic) -> dict:
    """Use Claude to extract the core prompt and intended output format.

    This implements the 'structured output' pattern:
    we instruct the model to respond exclusively with valid JSON,
    then parse and validate the result.

    Args:
        question: Raw user input (may contain format hints like 'draw', 'tell me').
        client:   Anthropic API client.

    Returns:
        Dict with keys:
            "prompt": str  — The cleaned educational question
            "format": str  — One of: "text", "audio", "image"
    """
    system = """\
You are an intent classifier for an educational assistant.
Analyse the user's input and respond ONLY with a JSON object — no other text.

JSON schema:
{
  "prompt": "<the educational question, cleaned of format hints>",
  "format": "<text | audio | image>"
}

Format detection rules:
- "image" if the user wants a drawing, illustration, or visual (draw, show, illustrate, нарисувай, покажи)
- "audio" if the user wants to hear the answer and uses phrases like: tell me, say it, read aloud, кажи ми, прочети, разкажи ми
- "text" for everything else (default)

Respond ONLY with the JSON object. No markdown, no explanation.
"""

    response = client.messages.create(
        model=cfg.CLAUDE_MODEL,
        max_tokens=200,
        system=system,
        messages=[{"role": "user", "content": question}]
    )

    raw = response.content[0].text.strip()

    # Robust JSON parsing — strip any accidental markdown fences
    raw = re.sub(r'^```json\s*|```$', '', raw, flags=re.MULTILINE).strip()

    try:
        parsed = json.loads(raw)
        # Validate format field
        if parsed.get("format") not in ("text", "audio", "image"):
            parsed["format"] = "text"
        return parsed
    except json.JSONDecodeError:
        # Graceful degradation: treat the whole question as a text request
        return {"prompt": question, "format": "text"}


# ─── Output Handlers ──────────────────────────────────────────────────────────

def render_text(answer: str) -> None:
    """Render the answer as formatted Markdown in the notebook."""
    display(Markdown(answer))


def render_audio(answer: str, language: str) -> None:
    """Convert the answer to speech and display an audio player.

    Uses Google Text-to-Speech (gTTS), which supports English and Bulgarian.
    The audio is saved to a temporary file and played inline.

    Engineering note:
        gTTS is a wrapper around Google's TTS API — it requires internet access.
        For offline / production use, consider Coqui TTS or ElevenLabs.

    Args:
        answer:   Text to convert to speech.
        language: Language code ("en" or "bg").
    """
    # Strip markdown formatting — TTS reads asterisks and hashes aloud
    clean_text = re.sub(r'[#*_`~>\[\]]', '', answer)
    clean_text = re.sub(r'📄|🌐', '', clean_text)   # Remove emoji markers
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    gtts_lang = cfg.AUDIO_LANG_MAP.get(language, "en")

    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        tmp_path = f.name

    tts = gTTS(text=clean_text, lang=gtts_lang, slow=False)
    tts.save(tmp_path)

    display(Markdown("🔊 **Audio Response:**"))
    display(Audio(tmp_path, autoplay=False))
    display(Markdown("---\n📝 **Transcript:**\n\n" + answer))


def render_image(answer: str, client: anthropic.Anthropic) -> None:
    """Generate an actual educational illustration using HuggingFace Inference API.

    Engineering note — two-step pipeline:
        Step 1 (Claude):  Convert the educational answer into an optimised
                          Stable Diffusion prompt (concise, keyword-rich,
                          with explicit style tokens).
        Step 2 (HF API):  Send that prompt to a hosted Stable Diffusion model
                          on HuggingFace and receive raw image bytes.

    Why HuggingFace Inference API?
        - Free tier available (requires only a HF token, no credit card)
        - No local GPU needed — model runs on HF servers
        - Dozens of SD models to choose from
        - Simple REST API — just POST a JSON body, receive image bytes

    Model choice — black-forest-labs/FLUX.1-schnell:
        - Reliable, well-supported, free tier compatible
        - Good at flat illustration styles with the right prompt tokens
        - Faster than SDXL for notebook use

    Args:
        answer: The text answer (used to derive the image prompt).
        client: Anthropic API client.
    """
    import requests
    import io
    from PIL import Image as PILImage

    # ── Step 0: Show the text answer first ───────────────────────────────────
    display(Markdown(answer))
    display(Markdown("---"))
    display(Markdown("🎨 **Generating illustration...** *(this takes ~15–30 seconds)*"))

    # ── Step 1: Ask Claude to write an optimised SD prompt ───────────────────
    # Engineering note:
    #   Stable Diffusion responds best to short, comma-separated keyword lists
    #   rather than full sentences. We explicitly request this format and add
    #   style tokens that push the model toward educational illustration output.
    prompt_response = client.messages.create(
        model=cfg.CLAUDE_MODEL,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": (
                "Write a Stable Diffusion image generation prompt for an educational "
                "illustration of the concept below. Use comma-separated keywords. "
                "Always include: flat illustration, bright colors, child-friendly, "
                "educational diagram, simple shapes, white background, clean style. "
                "Keep it under 60 words. Output ONLY the prompt, nothing else.\n\n"
                f"Concept:\n{answer[:400]}"
            )
        }]
    )
    sd_prompt = prompt_response.content[0].text.strip()

    # Negative prompt steers SD away from unwanted artefacts
    negative_prompt = (
        "blurry, dark, realistic photo, 3d render, ugly, deformed, "
        "text errors, watermark, signature, adult content"
    )

    display(Markdown(f"**Prompt sent to Stable Diffusion:**\n> `{sd_prompt}`"))

    # ── Step 2: Call HuggingFace Inference API ────────────────────────────────
    # Engineering note:
    #   The HF Inference API endpoint format is:
    #   https://api-inference.huggingface.co/models/{owner}/{model_name}
    #   It accepts JSON and returns raw image bytes (not base64, not JSON).
    #   We must set Accept: image/jpeg (or image/png) in the request headers.

    HF_API_URL = f"https://router.huggingface.co/hf-inference/models/{cfg.HF_MODEL}"

    headers = {
        "Authorization": f"Bearer {cfg.HF_TOKEN}",
        "Accept": "image/jpeg",
        "Content-Type": "application/json",
    }

    payload = {
        "inputs": sd_prompt,
        "parameters": {
            "negative_prompt": negative_prompt,
            "guidance_scale": 7.5,       # How strictly to follow the prompt
            "width": 512,
            "height": 512,
            "num_inference_steps": 30   # More steps = higher quality, slower
        }
    }

    try:

        response = requests.post(HF_API_URL, headers=headers, json=payload, timeout=120)

        if response.status_code == 503:
            # Model is loading — HF cold-starts models after inactivity
            display(Markdown(
                "⏳ **Model is loading on HuggingFace servers** (cold start ~20s). "
                "Please re-run this cell in 30 seconds."
            ))
            return

        if response.status_code == 401:
            display(Markdown(
                "❌ **Invalid HuggingFace token.** "
                "Check your `HF_TOKEN` in Section 3 or Colab Secrets."
            ))
            return

        if response.status_code != 200:
            display(Markdown(
                f"❌ **HuggingFace API error {response.status_code}:** "
                f"`{response.text[:300]}`"
            ))
            return

        # ── Decode and display the image ──────────────────────────────────────
        # The response body IS the image — just wrap it in BytesIO and open with Pillow
        image = PILImage.open(io.BytesIO(response.content))

        # Save to a temp file and display inline
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            tmp_img_path = f.name
        image.save(tmp_img_path)

        display(Markdown("✅ **Illustration generated:**"))
        display(Image(tmp_img_path, width=512))

    except requests.exceptions.Timeout:
        display(Markdown(
            "⏱️ **Request timed out.** HuggingFace free tier can be slow. "
            "Try again, or switch to a lighter model in `cfg.HF_MODEL`."
        ))
    except Exception as e:
        display(Markdown(f"❌ **Unexpected error:** `{e}`"))


# ─── Master Dispatcher: ask_ai() ─────────────────────────────────────────────

def ask_ai(
    question: str,
    collection: chromadb.Collection,
    embedding_model: SentenceTransformer,
    age_group: str,
    language: str,
    client: anthropic.Anthropic,
    verbose: bool = True
) -> None:
    """The top-level function: parse intent, retrieve, generate, and render.

    This is the single entry point for all user interactions.
    It orchestrates the full pipeline:
        1. Parse intent (format detection)
        2. Retrieve relevant document chunks
        3. Generate a grounded answer
        4. Route to the appropriate output handler

    Args:
        question:        The user's raw input.
        collection:      ChromaDB collection with document chunks.
        embedding_model: Loaded SentenceTransformer model.
        age_group:       Age group for style adaptation ("7-10", "10-16", "16+", "30+").
        language:        Response language ("en" or "bg").
        client:          Anthropic API client.
        verbose:         If True, shows retrieved chunks and intent parsing details.
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")

    # Step 1: Detect intent (format)
    intent = parse_intent(question, client)
    clean_question = intent["prompt"]
    output_format = intent["format"]

    if verbose:
        print(f"🎯 Intent parsed → format: {output_format} | clean prompt: {clean_question}")

    # Step 2 & 3: Retrieve context + generate answer
    answer = retrieve_information(
        question=clean_question,
        collection=collection,
        embedding_model=embedding_model,
        age_group=age_group,
        language=language,
        client=client,
        verbose=verbose
    )

    # Step 4: Route to the appropriate output handler
    print(f"\n📤 Output format: {output_format.upper()}\n")

    if output_format == "audio":
        render_audio(answer, language)
    elif output_format == "image":
        render_image(answer, client)
    else:
        render_text(answer)


---
## 📂 Section 8 — Document Upload & Indexing

Upload your PDF textbook here. The system will:
1. Extract all text, page by page
2. Split it into overlapping chunks
3. Embed every chunk with the local model
4. Store all embeddings in ChromaDB

**This only needs to run once per document.** After indexing, you can ask unlimited questions.


In [ ]:
# ─── Load the Embedding Model (run once) ─────────────────────────────────────
# This downloads ~80MB on first run, then uses the local cache.

embedding_model = load_embedding_model(cfg.EMBEDDING_MODEL)

In [ ]:
# ─── Upload PDF ───────────────────────────────────────────────────────────────
# A file chooser dialog will appear. Select your PDF textbook.

print("📎 Please upload your PDF file:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded. Please run this cell again and select a PDF.")

pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {pdf_filename} ({len(uploaded[pdf_filename]) / 1024:.1f} KB)")

In [ ]:
# ─── Process & Index the Document ────────────────────────────────────────────
# This cell runs the full indexing pipeline:
#   PDF → text pages → chunks → embeddings → ChromaDB

print("🔄 Starting document indexing pipeline...\n")

# Step 1: Extract text from PDF
pages = extract_text_from_pdf(pdf_filename)

# Step 2: Chunk the text
chunks = chunk_text(pages, chunk_size=cfg.CHUNK_SIZE, overlap=cfg.CHUNK_OVERLAP)

# Step 3: Embed chunks and build the vector store
collection = build_vector_store(chunks, embedding_model, cfg.COLLECTION_NAME)

print("\n🎉 Document indexed successfully! You can now ask questions.")

---
## 🎓 Section 9 — User Setup

Configure the assistant for the student's age group and preferred language.


In [ ]:
# ─── Student Configuration ────────────────────────────────────────────────────
# Modify the two variables below before asking questions.

# Age group options: "7-10" | "10-16" | "16+" | "30+"
AGE_GROUP = "10-16"

# Language options: "en" (English) | "bg" (Bulgarian)
LANGUAGE = "en"

print(f"✅ Student profile set:")
print(f"   Age group : {AGE_GROUP}")
print(f"   Language  : {'English' if LANGUAGE == 'en' else 'Bulgarian'}")
print(f"\nStyle: {AGE_STYLE_MAP[AGE_GROUP][:100]}...")

---
## 💬 Section 10 — Ask Questions

Now you can ask anything! Try different formats:

| Intent | Example |
|---|---|
| Text answer | `"What is photosynthesis?"` |
| Audio answer | `"Tell me about the water cycle"` |
| Illustration prompt | `"Draw how a cell divides"` |

Set `verbose=True` to see the retrieved chunks and intent detection in action — great for learning how RAG works!


In [ ]:
# ─── Ask a Question ───────────────────────────────────────────────────────────
# Change the question below and re-run this cell.

ask_ai(
    question="представи графично икономическите фактори отнасящи се за Япония?",
    collection=collection,
    embedding_model=embedding_model,
    age_group=AGE_GROUP,
    language=LANGUAGE,
    client=claude,
    verbose=False   # Set to True to see the RAG pipeline in action
)

---
## 🧪 Section 11 — Test Suite

### Engineering Note: Why Test RAG Systems?

RAG systems have multiple failure modes:
- **Retrieval failure**: The right chunk isn't retrieved (embedding quality issue)
- **Generation failure**: The LLM ignores the context or hallucinates
- **Format routing failure**: Wrong output modality is selected
- **Fallback failure**: The system doesn't handle missing information gracefully

A comprehensive test suite catches these failures early and documents expected behaviour.

The 8 tests below cover all critical paths of the system.


In [ ]:
# ─── Test Runner Infrastructure ───────────────────────────────────────────────

def run_test(
    test_name: str,
    question: str,
    age_group: str,
    language: str,
    expected_format: Optional[str] = None,
    verbose: bool = False
) -> bool:
    """Run a single named test and report the result.

    Args:
        test_name:       Human-readable test description.
        question:        The question to ask.
        age_group:       Age group for the test.
        language:        Language for the test.
        expected_format: If set, verifies the detected output format.
        verbose:         If True, shows full pipeline details.

    Returns:
        True if the test passed, False otherwise.
    """
    print(f"\n{'─'*60}")
    print(f"🧪 TEST: {test_name}")
    print(f"{'─'*60}")

    try:
        # Check format detection if expected_format is specified
        if expected_format:
            intent = parse_intent(question, claude)
            detected = intent["format"]
            format_ok = detected == expected_format
            status = "✅" if format_ok else "❌"
            print(f"{status} Format detection: expected={expected_format}, got={detected}")
            if not format_ok:
                return False

        # Run the full pipeline
        ask_ai(
            question=question,
            collection=collection,
            embedding_model=embedding_model,
            age_group=age_group,
            language=language,
            client=claude,
            verbose=verbose
        )

        print(f"\n✅ Test passed: {test_name}")
        return True

    except Exception as e:
        print(f"\n❌ Test FAILED: {test_name}")
        print(f"   Error: {e}")
        return False

In [ ]:
# ─── Run All 8 Tests ──────────────────────────────────────────────────────────
# These tests cover the full spectrum of system behaviour.

test_results = {}

# ── Test 1: Simple Factual Question → Text Output ────────────────────────────
# Validates: basic retrieval + text generation for a young audience
test_results["T1"] = run_test(
    test_name="T1: Simple factual question (age 7-10, English, text)",
    question="What is the most important thing explained in this document?",
    age_group="7-10",
    language="en",
    expected_format="text"
)

# ── Test 2: Complex Explanation → Text Output ────────────────────────────────
# Validates: multi-chunk retrieval for a nuanced question
test_results["T2"] = run_test(
    test_name="T2: Complex explanation request (age 16+, English, text)",
    question="Can you explain how the main concepts in this document are interconnected?",
    age_group="16+",
    language="en",
    expected_format="text"
)

# ── Test 3: Audio Format Detection ───────────────────────────────────────────
# Validates: 'tell me' → audio format routing
test_results["T3"] = run_test(
    test_name="T3: Audio output trigger ('tell me', age 10-16, English)",
    question="Tell me about the key ideas in this document",
    age_group="10-16",
    language="en",
    expected_format="audio"
)

# ── Test 4: Image Format Detection ───────────────────────────────────────────
# Validates: 'draw' → image format routing
test_results["T4"] = run_test(
    test_name="T4: Image output trigger ('draw', age 7-10, English)",
    question="Draw an illustration showing the main concept from this document",
    age_group="7-10",
    language="en",
    expected_format="image"
)

# ── Test 5: Missing Information Fallback ─────────────────────────────────────
# Validates: the model gracefully handles questions not in the document
test_results["T5"] = run_test(
    test_name="T5: Fallback for missing information",
    question="What is the population of Mars and who lives there?",
    age_group="10-16",
    language="en",
    expected_format="text"
)

# ── Test 6: Mixed Answer (PDF + AI knowledge) ────────────────────────────────
# Validates: source attribution markers appear correctly
test_results["T6"] = run_test(
    test_name="T6: Mixed source answer (document + AI knowledge)",
    question="Summarise what this document covers and add any related context you know",
    age_group="16+",
    language="en",
    expected_format="text"
)

# ── Test 7: Bulgarian Language ────────────────────────────────────────────────
# Validates: full pipeline works correctly in Bulgarian
test_results["T7"] = run_test(
    test_name="T7: Bulgarian language response",
    question="Какво е най-важното нещо, обяснено в този документ?",
    age_group="10-16",
    language="bg",
    expected_format="text"
)

# ── Test 8: Adult Learner Style ───────────────────────────────────────────────
# Validates: age 30+ produces a professional, concise tone
test_results["T8"] = run_test(
    test_name="T8: Adult learner (age 30+, English, concise professional tone)",
    question="Give me a concise professional summary of this document's core content",
    age_group="30+",
    language="en",
    expected_format="text"
)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for v in test_results.values() if v)
total = len(test_results)
print(f"\n{'='*60}")
print(f"📊 TEST SUMMARY: {passed}/{total} passed")
print(f"{'='*60}")
for test_id, result in test_results.items():
    icon = "✅" if result else "❌"
    print(f"  {icon} {test_id}")

---
## 🔬 Section 12 — Engineering Deep Dives

These cells are for **engineering students** who want to understand the system's internals.
They are not required for the end-user experience.


In [ ]:
# ─── Deep Dive 1: Inspect Chunks ──────────────────────────────────────────────
# See exactly how the document was split into chunks.
# This helps understand how chunk size affects retrieval quality.

print(f"Total chunks in vector store: {collection.count()}")
print(f"\nFirst 3 chunks:\n")

for chunk in chunks[:3]:
    print(f"ID: {chunk['chunk_id']} | Page: {chunk['page']}")
    print(f"Length: {len(chunk['text'])} chars (~{len(chunk['text'])//4} tokens)")
    print(f"Preview: {chunk['text'][:200]}...")
    print("─" * 50)

In [ ]:
# ─── Deep Dive 2: Visualise Retrieval ────────────────────────────────────────
# See which chunks are retrieved for a given query and their similarity scores.
# This reveals how the vector search decides what's "relevant".

test_query = "Explain the main concept"   # ← Change this to test different queries

print(f"Query: '{test_query}'\n")
retrieved = retrieve_chunks(test_query, collection, embedding_model, top_k=cfg.TOP_K)

for i, chunk in enumerate(retrieved, 1):
    bar_length = int(chunk['similarity'] * 40)
    bar = '█' * bar_length + '░' * (40 - bar_length)

    print(f"Chunk {i} | Page {chunk['page']} | Similarity: {chunk['similarity']:.2%}")
    print(f"[{bar}]")
    print(f"{chunk['text'][:200]}...")
    print()

In [ ]:
# ─── Deep Dive 3: Inspect Embedding Vectors ───────────────────────────────────
# Peek at the actual numbers inside an embedding.
# This demystifies what a 'vector' really is.

sample_text = "This is an educational example sentence."
embedding = embedding_model.encode(sample_text)

print(f"Input text: '{sample_text}'")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"Embedding type: {embedding.dtype}")
print(f"\nFirst 10 values (out of {len(embedding)}):")
print([round(float(v), 4) for v in embedding[:10]])
print(f"\nValue range: [{float(embedding.min()):.4f}, {float(embedding.max()):.4f}]")
print(f"\nEngineering insight: Each of the {len(embedding)} numbers encodes a\n"
      f"different semantic feature of the text. Similar sentences will have\n"
      f"similar patterns across all {len(embedding)} dimensions.")

In [ ]:
# ─── Deep Dive 4: Cosine Similarity from Scratch ─────────────────────────────
# Understand why cosine similarity works better than Euclidean distance
# for comparing text embeddings.

import numpy as np

def cosine_similarity(a: list, b: list) -> float:
    """Compute cosine similarity between two vectors.

    Formula: cos(θ) = (A · B) / (|A| × |B|)

    This measures the ANGLE between two vectors, not their distance.
    Two sentences can have very different word counts but nearly identical
    meaning — cosine similarity captures this, Euclidean distance does not.
    """
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Compare pairs of sentences
pairs = [
    ("A dog runs in the park", "A canine sprints through the garden"),
    ("The sky is blue", "Machine learning is fascinating"),
    ("Photosynthesis converts sunlight to energy", "Plants use light to make food"),
]

print("Cosine Similarity Demonstration\n" + "="*50)
for s1, s2 in pairs:
    e1 = embedding_model.encode(s1)
    e2 = embedding_model.encode(s2)
    sim = cosine_similarity(e1, e2)
    bar = '█' * int(sim * 30)
    print(f"\nA: '{s1}'")
    print(f"B: '{s2}'")
    print(f"Similarity: {sim:.4f} [{bar}]")

---
## 🚀 Section 13 — Future Extensions

This section documents improvements for engineering students who want to extend the system.

### 1. Query Rewriting
Before retrieval, use Claude to rephrase the question for better embedding match:
```python
def rewrite_query(question: str, client) -> str:
    # Ask Claude: "Rephrase this as a precise factual query: {question}"
    # Returns a cleaner, more searchable version
```

### 2. Hybrid Retrieval
Combine dense (vector) search with sparse (BM25 keyword) search for better coverage:
```python
# Dense results: semantically similar chunks
# Sparse results: exact keyword matches
# Final: Reciprocal Rank Fusion to merge both
```

### 3. Conversation Memory
Maintain conversation history so follow-up questions work correctly:
```python
conversation_history = []
# Include last N turns in the LLM prompt
# This allows: "Tell me more about the second point you mentioned"
```

### 4. Persistent Vector Store
Replace in-memory ChromaDB with disk-backed storage:
```python
chroma_client = chromadb.PersistentClient(path="./chroma_db")
# Now the index survives runtime restarts
```

### 5. Production Deployment
- **Backend**: FastAPI with async endpoints
- **Frontend**: React with streaming responses (Server-Sent Events)
- **Storage**: PostgreSQL + pgvector for production-scale vector storage
- **Auth**: JWT tokens for student session management


In [ ]:
# ─── Bonus: Query Rewriting (Extension Implementation) ────────────────────────
# This implements the first future extension — a practical example
# of how to improve retrieval quality with minimal additional code.

def rewrite_query_for_retrieval(question: str, client: anthropic.Anthropic) -> str:
    """Rewrite a conversational question into a precise retrieval query.

    Engineering insight:
        Conversational questions contain filler words and implicit references
        that hurt embedding quality. A rewritten, fact-focused query matches
        document chunks much more precisely.

        Example:
            Input:  "Can you explain what the thing about energy transfer is?"
            Output: "Energy transfer process definition and mechanism"

    Args:
        question: The user's original question.
        client:   Anthropic API client.

    Returns:
        A concise, keyword-rich retrieval query.
    """
    response = client.messages.create(
        model=cfg.CLAUDE_MODEL,
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": (
                f"Rewrite this question as a concise, keyword-rich search query "
                f"for a document retrieval system. Output ONLY the query, nothing else.\n\n"
                f"Question: {question}"
            )
        }]
    )
    return response.content[0].text.strip()


# Demonstration
original = "Can you explain what that thing about how plants make food is all about?"
rewritten = rewrite_query_for_retrieval(original, claude)

print("Query Rewriting Demonstration")
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")
print("\n💡 The rewritten query will match document chunks much more accurately.")

---

## 📋 Quick Reference Card

| Function | Purpose | Key Arguments |
|---|---|---|
| `extract_text_from_pdf(path)` | Extract text page by page | `pdf_path` |
| `chunk_text(pages, size, overlap)` | Split into overlapping chunks | `chunk_size`, `overlap` |
| `load_embedding_model(name)` | Load SentenceTransformer | `model_name` |
| `build_vector_store(chunks, model, name)` | Embed + store in ChromaDB | `chunks`, `embedding_model` |
| `retrieve_chunks(query, collection, model, k)` | Top-k similarity search | `query`, `top_k` |
| `retrieve_information(...)` | Full RAG pipeline | `question`, `age_group`, `language` |
| `parse_intent(question, client)` | Detect output format | `question` |
| `ask_ai(...)` | Master dispatcher | All of the above |

---

*Built with ❤️ using the Claude API · ChromaDB · SentenceTransformers*
